# 02. 지하철 피처 엔지니어링 — 서울 아파트 최근 5년

[01_base_eda_feature_engineering.ipynb](01_base_eda_feature_engineering.ipynb)에서 만든
`df_fe_base.parquet`(전처리 + 기본/시간 피처 + 시장 lag 피처 + 좌표)을 불러와
최근접 지하철역 거리 피처를 추가합니다.

학교/유치원 피처는 별도로 [03_school_feature_engineering.ipynb](03_school_feature_engineering.ipynb)에서
같은 base 결과를 불러와 만들며, 두 결과의 병합은 이후 단계에서 진행합니다.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


# ------------------------------------------------------------------
# 경로 설정
# ------------------------------------------------------------------

# 현재 위치:
# Apartment price/notebook/v2_5year_notebook/eda
PROJECT_DIR = Path.cwd().parents[2]

DATA_ROOT = PROJECT_DIR / "data"

# V2 서울 5년 데이터
DATA_DIR = DATA_ROOT / "v2_5year_seoul"

PROCESSED_DIR = DATA_DIR / "processed"

# V1 / V2 공통 외부 데이터
EXTERNAL_DIR = DATA_ROOT / "external"


# ------------------------------------------------------------------
# 01_base_eda_feature_engineering.ipynb 결과 로드
# ------------------------------------------------------------------

DF_FE_BASE_PATH = PROCESSED_DIR / "df_fe_base.parquet"

if not DF_FE_BASE_PATH.exists():
    raise FileNotFoundError(
        f"기본 피처 데이터가 없습니다.\n"
        f"경로: {DF_FE_BASE_PATH}\n"
        f"먼저 01_base_eda_feature_engineering.ipynb를 실행하세요."
    )

df_fe = pd.read_parquet(DF_FE_BASE_PATH)

print("[입력 데이터]")
print("DF_FE_BASE_PATH :", DF_FE_BASE_PATH.resolve())
print("df_fe shape     :", df_fe.shape)


[입력 데이터]
DF_FE_BASE_PATH : C:\Users\UK\Apartment price\data\v2_5year_seoul\processed\df_fe_base.parquet
df_fe shape     : (234302, 69)


In [2]:
# ============================================================
# 기본 데이터 및 transaction_id 확인
# ============================================================

print("전체 거래 수:", len(df_fe))
print(
    "transaction_id 존재:",
    "transaction_id" in df_fe.columns
)
print(
    "transaction_id 고유 개수:",
    df_fe["transaction_id"].nunique()
)
print(
    "transaction_id 중복:",
    df_fe["transaction_id"].duplicated().sum()
)

전체 거래 수: 234302
transaction_id 존재: True
transaction_id 고유 개수: 234302
transaction_id 중복: 0


### 6-1. 지하철 거리 계산 전 전처리 완료 확인

여기까지가 **거래 데이터 전처리 + 공통 피처 엔지니어링 + 시장 lag 피처 + 아파트 좌표 매핑** 단계입니다.

아래 검증을 통과한 뒤부터 지하철·교육시설 등 외부 생활 인프라 피처를 추가합니다.


In [3]:
# ============================================================
# 지하철 거리 계산 전 최종 검증
# ============================================================

required_preprocessed_cols = [
    "거래월일자",
    "경과월",
    "연식",
    "평당가",
    "log_평당가",
    "log_price",
    "시군구명",
    "구_전월_중위가격",
    "서울_전월_중위가격",
    "검색주소",
    "위도",
    "경도",
]

missing_cols = [
    col
    for col in required_preprocessed_cols
    if col not in df_fe.columns
]

remaining_cancelled = int(
    df_fe["해제여부"]
    .eq("O")
    .fillna(False)
    .sum()
)

coord_missing = int(
    df_fe[["위도", "경도"]]
    .isna()
    .any(axis=1)
    .sum()
)

if missing_cols:
    raise ValueError(
        f"전처리 필수 컬럼 누락: {missing_cols}"
    )

if remaining_cancelled > 0:
    raise ValueError(
        f"해제 거래가 {remaining_cancelled:,}건 남아 있습니다."
    )

print("[전처리 완료]")
print(f"최종 거래 수     : {len(df_fe):,}")
print(f"해제 거래 잔여   : {remaining_cancelled:,}")
print(f"좌표 결측        : {coord_missing:,}")
print(f"거래 월 수       : {df_fe['거래월일자'].dt.to_period('M').nunique()}")
print("→ 지하철/교육 인프라 피처 생성 단계로 진행 가능")


[전처리 완료]
최종 거래 수     : 234,302
해제 거래 잔여   : 0
좌표 결측        : 0
거래 월 수       : 60
→ 지하철/교육 인프라 피처 생성 단계로 진행 가능


## 7. 최근접 지하철역 거리

아파트와 지하철역의 위도·경도를 기반으로
`sklearn.neighbors.BallTree`의 `haversine` 거리를 사용하여
각 아파트에서 가장 가까운 지하철역을 탐색합니다.

최근접 역까지의 거리는 지구 반지름을 곱해 미터(m) 단위의
`역세권거리_m` 피처로 생성합니다.

추가로 최근접 역의 이름과 노선 정보를 저장하여
입지 분석 및 웹 서비스 설명에 활용합니다.

### 5년 데이터에서의 시간적 주의사항

현재 `subway_stations.xlsx`가 **현재 시점의 역 목록**이라면,
5년 데이터 기간 중 새로 개통한 역이 과거 거래 시점에도
존재했던 것처럼 계산될 수 있습니다.

따라서 현재 버전의 `역세권거리_m`은
**현재 기준 입지 proxy**로 사용합니다.

향후 역 개통일 데이터가 확보되면
거래 시점에 실제 운영 중이었던 역만 후보에 포함하여

`거래일자 >= 역 개통일`

조건을 적용한 시점별 최근접 거리로 고도화할 수 있습니다.

In [4]:
subway_df = pd.read_excel(EXTERNAL_DIR / "subway_stations.xlsx")
print(subway_df.shape)
subway_df[["역사명", "노선명", "역위도", "역경도", "운영기관명"]].head()


(1099, 15)


,역사명,노선명,역위도,역경도,운영기관명
0,신사,신분당선,37.516125,127.019761,경기도 신분당선
1,논현,신분당선,37.511115,127.021366,경기도 신분당선
2,신논현,신분당선,37.504555,127.024531,경기도 신분당선
3,강남,신분당선,37.497385,127.027879,경기도 신분당선
4,양재(서초구청),신분당선,37.484454,127.034099,경기도 신분당선


지하철역 데이터는 1,099개 역(수도권 전체 노선 포함으로 보임) 기준이며, 위경도 좌표가 포함되어 있어 바로 `nearest_facility` 함수에 사용할 수 있습니다.

In [5]:
from sklearn.neighbors import BallTree

EARTH_RADIUS_M = 6_371_000

apartment_coords = np.radians(
    df_fe[["위도", "경도"]].to_numpy()
)

subway_coords = np.radians(
    subway_df[["역위도", "역경도"]].to_numpy()
)

subway_tree = BallTree(
    subway_coords,
    metric="haversine"
)

distances, indices = subway_tree.query(
    apartment_coords,
    k=1
)

nearest_idx = indices[:, 0]

df_fe["역세권거리_m"] = (
    distances[:, 0] * EARTH_RADIUS_M
)

df_fe["최근접역"] = (
    subway_df.iloc[nearest_idx]["역사명"]
    .to_numpy()
)

df_fe["최근접노선"] = (
    subway_df.iloc[nearest_idx]["노선명"]
    .to_numpy()
)

역세권거리는 중앙값 약 499m, 평균 약 760m로 우측 꼬리가 긴 분포입니다(최댓값 약 9,328m). 대부분의 서울 아파트가 도보권(500m 내외)에 지하철역을 두고 있다는 점과 부합합니다.

In [6]:
# ============================================================
# 지하철 거리 이상치 확인
# ============================================================

farthest_apartments = (
    df_fe[
        [
            "법정동",
            "단지명",
            "최근접역",
            "역세권거리_m",
        ]
    ]
    .sort_values(
        "역세권거리_m",
        ascending=False
    )
    .head(5)
)

display(farthest_apartments)


# ============================================================
# 지하철 거리 분포 확인
# ============================================================

print(
    df_fe["역세권거리_m"].describe(
        percentiles=[
            0.5,
            0.75,
            0.9,
            0.95,
            0.99,
        ]
    )
)

,법정동,단지명,최근접역,역세권거리_m
499,평창동,일성빌라트,북한산보국문,3143.458506
382,평창동,일성빌라트,북한산보국문,3143.458506
663,평창동,일성빌라트,북한산보국문,3143.458506
1402,평창동,일성빌라트,북한산보국문,3143.458506
2045,평창동,일성빌라트,북한산보국문,3143.458506


count    234302.000000
mean        531.752437
std         321.697386
min          24.436073
50%         461.863773
75%         659.260223
90%         928.849431
95%        1174.163303
99%        1714.925153
max        3143.458506
Name: 역세권거리_m, dtype: float64


In [7]:
# ============================================================
# 역세권 거리 구간
# ============================================================

def classify_subway_distance(distance):
    if distance <= 500:
        return "500m 이내"
    elif distance <= 1000:
        return "500m~1km"
    elif distance <= 1500:
        return "1km~1.5km"
    return "1.5km 초과"


df_fe["역세권_구간"] = (
    df_fe["역세권거리_m"]
    .apply(classify_subway_distance)
)

In [8]:
subway_segment = (
    df_fe["역세권_구간"]
    .value_counts()
    .to_frame("거래건수")
)

subway_segment["비율"] = (
    subway_segment["거래건수"]
    / len(df_fe)
)

display(subway_segment)

,거래건수,비율
역세권_구간,,
500m 이내,129660,0.553388
500m~1km,86088,0.367423
1km~1.5km,14293,0.061002
1.5km 초과,4261,0.018186


In [9]:
farthest_apartments = (
    df_fe[
        [
            "시군구명",
            "법정동",
            "단지명",
            "최근접역",
            "역세권거리_m",
        ]
    ]
    .sort_values(
        "역세권거리_m",
        ascending=False
    )
    .head(5)
)

display(farthest_apartments)

,시군구명,법정동,단지명,최근접역,역세권거리_m
499,종로구,평창동,일성빌라트,북한산보국문,3143.458506
382,종로구,평창동,일성빌라트,북한산보국문,3143.458506
663,종로구,평창동,일성빌라트,북한산보국문,3143.458506
1402,종로구,평창동,일성빌라트,북한산보국문,3143.458506
2045,종로구,평창동,일성빌라트,북한산보국문,3143.458506


In [11]:
subway_feature_cols = [
    "transaction_id",
    "최근접역",
    "최근접노선",
    "역세권거리_m",
    "역세권_구간",
]

print("누락 컬럼:", [
    col for col in subway_feature_cols
    if col not in df_fe.columns
])

print("\n결측치:")
print(df_fe[subway_feature_cols].isna().sum())

display(df_fe[subway_feature_cols].head())

누락 컬럼: []

결측치:
transaction_id    0
최근접역              0
최근접노선             0
역세권거리_m           0
역세권_구간            0
dtype: int64


,transaction_id,최근접역,최근접노선,역세권거리_m,역세권_구간
0,0,독립문,3호선,303.869432,500m 이내
1,1,경복궁(정부서울청사),3호선,443.144690,500m 이내
2,2,서대문,5호선,605.216200,500m~1km
3,3,경복궁(정부서울청사),3호선,377.021196,500m 이내
4,4,창신,6호선,311.066828,500m 이내


In [13]:
# ============================================================
# 지하철 피처 데이터 저장
# ============================================================

DF_FE_SUBWAY_PATH = (
    PROCESSED_DIR / "df_fe_subway.parquet"
)

subway_feature_cols = [
    "transaction_id",
    "최근접역",
    "최근접노선",
    "역세권거리_m",
    "역세권_구간",
]

df_fe_subway = (
    df_fe[subway_feature_cols]
    .copy()
)

# 저장 전 검증
assert len(df_fe_subway) == len(df_fe)
assert df_fe_subway["transaction_id"].is_unique

df_fe_subway.to_parquet(
    DF_FE_SUBWAY_PATH,
    index=False,
)

print("[지하철 피처 저장 완료]")
print("저장 경로:", DF_FE_SUBWAY_PATH.resolve())
print("저장 shape:", df_fe_subway.shape)

display(df_fe_subway.head())

[지하철 피처 저장 완료]
저장 경로: C:\Users\UK\Apartment price\data\v2_5year_seoul\processed\df_fe_subway.parquet
저장 shape: (234302, 5)


,transaction_id,최근접역,최근접노선,역세권거리_m,역세권_구간
0,0,독립문,3호선,303.869432,500m 이내
1,1,경복궁(정부서울청사),3호선,443.144690,500m 이내
2,2,서대문,5호선,605.216200,500m~1km
3,3,경복궁(정부서울청사),3호선,377.021196,500m 이내
4,4,창신,6호선,311.066828,500m 이내
